Aim of this script: identify the rail operator for each row, using the `data_chuuchuu_{data_selection}_terminus.parquet` intermediate output produced by `Chuuchuu_data_test_terminus_identification.ipynb`.

In [21]:
import pandas as pd
import numpy as np
import os

In [22]:
data_selection = "combined"

intermediate_outputs_dir = "intermediate_outputs"
data_path = f"{intermediate_outputs_dir}/data_chuuchuu_{data_selection}_terminus.parquet"

try: 
    data_chuuchuu.head()
except NameError:
    data_chuuchuu = pd.read_parquet(data_path)
data_chuuchuu.shape

(15000614, 36)

### Helper columns & constants

Translating the SQL rules from `instructions_chuuchuu_operator_identification.md` into pandas:

- `UPPER(TRIM(col))` comparisons -> normalized `_norm` columns.
- `routeNumber` list/CAST(...AS INTEGER) BETWEEN comparisons -> a trimmed string column plus a numeric column (non-numeric values become `NaN` and simply won't match any range check).
- `stopcountry = 'XX'` -> the full country name already resolved in this dataset's `country` column (see `Chuuchuu_data_test_country_attribution.ipynb`).

Rules are applied in the same order as the instructions doc, so later rules intentionally overwrite `operator` for rows also matched by an earlier, broader rule -- exactly like running the SQL `UPDATE`s in sequence.

In [23]:
def norm(series):
    """Mirror SQL's UPPER(TRIM(...)) comparison."""
    return series.astype(str).str.strip().str.upper()

def in_ranges(int_series, ranges):
    """OR together several inclusive (lo, hi) BETWEEN checks; NaN (non-numeric routeNumber) never matches."""
    mask = pd.Series(False, index=int_series.index)
    for lo, hi in ranges:
        mask |= int_series.between(lo, hi)
    return mask

data_chuuchuu["routeType_norm"] = norm(data_chuuchuu["routeType"])
data_chuuchuu["agency_norm"] = norm(data_chuuchuu["agency"])
data_chuuchuu["routeNumber_trim"] = data_chuuchuu["routeNumber"].astype(str).str.strip()
data_chuuchuu["routeNumber_int"] = pd.to_numeric(data_chuuchuu["routeNumber_trim"], errors="coerce")

print(f"{data_chuuchuu['routeNumber_int'].isna().sum()} rows have a non-numeric routeNumber (can't be matched by any routeNumber-range rule below)")

122691 rows have a non-numeric routeNumber (can't be matched by any routeNumber-range rule below)


### Highspeed services

In [24]:
# routeType matched with its exact (case-sensitive) literal, same as the source SQL
highspeed_operator_by_routeType = {
    "ICE": "Deutsche Bahn",
    "TGV": "SNCF", "TGV INOUI": "SNCF", "TGV Lyria": "SNCF", "LYR": "SNCF", "LYRIA": "SNCF", "Ouigo": "SNCF", "OUI": "SNCF",
    "RJX": "OEBB", "rjx": "OEBB",
    "FR": "Trenitalia",
    "EST": "Eurostar", "EUR": "Eurostar", "Eurostar": "Eurostar",
    "Italo": "Italo",
}

highspeed_mask = data_chuuchuu["routeType"].isin(highspeed_operator_by_routeType)
data_chuuchuu.loc[highspeed_mask, "operator"] = data_chuuchuu.loc[highspeed_mask, "routeType"].map(highspeed_operator_by_routeType)
print(f"{highspeed_mask.sum()} rows assigned an operator via the highspeed routeType rule")

268761 rows assigned an operator via the highspeed routeType rule


### Night train services

`NJ`/`Nightjet` -> OEBB, `ES`/`European Sleeper` -> European Sleeper, and `EN` split by `routeNumber` (a handful of `EN` route numbers are first relabeled to `NJ`, per the instructions).

In [25]:
# relabel the NJ-under-EN route numbers first, so the NJ operator rule below picks them up too
en_to_nj_route_numbers = {"294", "295", "13485", "233"}
en_to_nj_mask = (data_chuuchuu["routeType_norm"] == "EN") & data_chuuchuu["routeNumber_trim"].isin(en_to_nj_route_numbers)
data_chuuchuu.loc[en_to_nj_mask, "routeType"] = "NJ"
data_chuuchuu.loc[en_to_nj_mask, "routeType_norm"] = "NJ"
print(f"{en_to_nj_mask.sum()} rows had routeType 'EN' relabeled to 'NJ'")

data_chuuchuu.loc[data_chuuchuu["routeType_norm"].isin(["ES", "EUROPEAN SLEEPER"]), "operator"] = "European Sleeper"
data_chuuchuu.loc[data_chuuchuu["routeType_norm"].isin(["NJ", "NIGHTJET"]), "operator"] = "OEBB"

en_operator_by_routeNumber = {
    "SJ": {"344", "345", "346", "13471", "13472"},
    "HZ": {"40465", "40414", "40237", "414", "415"},
    "PKP": {"406", "407", "40417", "40416", "40407", "1276", "1277"},
    "Ceske Drahy": {"40458", "40459", "443", "442"},
    "MAV": {"40462", "40467", "50237", "50462", "40476", "40457", "40406", "462", "476", "477", "463"},
}
for operator_name, route_numbers in en_operator_by_routeNumber.items():
    mask = (data_chuuchuu["routeType_norm"] == "EN") & data_chuuchuu["routeNumber_trim"].isin(route_numbers)
    data_chuuchuu.loc[mask, "operator"] = operator_name
    print(f"{mask.sum()} EN rows assigned operator '{operator_name}'")

# route numbers the instructions explicitly flag as having no known operator yet -- left untouched
en_unknown_route_numbers = {
    "1415", "1153", "1152", "50476", "323",
    "13400", "13403", "13451", "13408", "13401", "13402", "13404", "13406", "13420", "13405", "13409", "13417",
    "93701", "319", "580", "230", "320", "34834", "91641", "91505", "32",
    "277", "37501", "89843", "89962", "11799", "28565", "28960", "370", "20159",
}
en_unknown_mask = (data_chuuchuu["routeType_norm"] == "EN") & data_chuuchuu["routeNumber_trim"].isin(en_unknown_route_numbers)
print(f"{en_unknown_mask.sum()} EN rows have no known operator per the instructions -- left as-is")

0 rows had routeType 'EN' relabeled to 'NJ'
164 EN rows assigned operator 'SJ'
2258 EN rows assigned operator 'HZ'
1652 EN rows assigned operator 'PKP'
527 EN rows assigned operator 'Ceske Drahy'
3350 EN rows assigned operator 'MAV'
18 EN rows have no known operator per the instructions -- left as-is


### Railjet (ÖBB or Ceske Drahy)

In [26]:
ceske_drahy_rj_route_numbers = {
    "50", "51", "52", "53", "54", "55", "56",
    "70", "71", "72", "73", "74", "75", "78", "79",
    "170", "171", "172", "173", "174", "175", "176", "177", "178", "179",
    "244", "250", "251", "252", "253", "254", "255", "256", "257", "258", "259",
    "270", "271", "272", "273", "274", "275", "276", "277",
    "284", "285",
    "370", "371", "372", "373", "374", "375",
    "382", "383", "384", "385", "386", "387",
    "478", "479",
    "512", "515",
    "548", "549",
    "576", "577", "578", "579",
    "644", "645",
}

rj_mask = data_chuuchuu["routeType_norm"] == "RJ"
is_ceske_drahy_rj = rj_mask & data_chuuchuu["routeNumber_trim"].isin(ceske_drahy_rj_route_numbers)
data_chuuchuu.loc[is_ceske_drahy_rj, "operator"] = "Ceske Drahy"
data_chuuchuu.loc[rj_mask & ~is_ceske_drahy_rj, "operator"] = "OEBB"
print(f"{is_ceske_drahy_rj.sum()} RJ rows assigned 'Ceske Drahy', {(rj_mask & ~is_ceske_drahy_rj).sum()} assigned 'OEBB'")

6981 RJ rows assigned 'Ceske Drahy', 18483 assigned 'OEBB'


### Easy operator cases (PL, FLX, HU, IT, FR)

In [27]:
pl_mask = data_chuuchuu["agency"] == "PL"
data_chuuchuu.loc[pl_mask, "operator"] = "PKP Intercity"

flx_mask = data_chuuchuu["routeType_norm"] == "FLX"
data_chuuchuu.loc[flx_mask, "operator"] = "Flixtrain"

hu_routeTypes = {"EC", "IC", "EX", "EN", "G", "GY", "ER", "H", "IR", "S", "SZ", "Z"}
hu_mask = (data_chuuchuu["agency_norm"] == "HU") & data_chuuchuu["routeType_norm"].isin(hu_routeTypes)
data_chuuchuu.loc[hu_mask, "operator"] = "MAV"

it_routeTypes = {"FR", "FA", "FB", "IC", "ICN", "EXP", "IR", "REG", "MET", "EN", "NCL"}
it_mask = (data_chuuchuu["agency_norm"] == "IT") & data_chuuchuu["routeType_norm"].isin(it_routeTypes)
data_chuuchuu.loc[it_mask, "operator"] = "Trenitalia"

fr_routeTypes = {
    "IC", "ICN", "INTERCITES", "INTERCITES DE NUIT", "LYR", "LYRIA",
    "NAV", "NAVETTE", "OGO", "OUI", "OUIGO", "TER", "TGV INOUI", "TRAIN TER",
}
fr_mask = (data_chuuchuu["agency_norm"] == "FR") & data_chuuchuu["routeType_norm"].isin(fr_routeTypes)
data_chuuchuu.loc[fr_mask, "operator"] = "SNCF"

print(f"PL: {pl_mask.sum()}, FLX: {flx_mask.sum()}, HU: {hu_mask.sum()}, IT: {it_mask.sum()}, FR: {fr_mask.sum()}")

PL: 57049, FLX: 2469, HU: 733369, IT: 1479286, FR: 1021073


### SBB agency

`IC`/`EC` -> SBB whenever the stop is in Switzerland. `IR`/`R`/`RE`/`S`/`SN` are only *sometimes* SBB, split by `routeNumber` blocks. `PE`/`ICE`/`RB` have no rule.

In [28]:
# IC, EC inside Switzerland -> SBB
sbb_ic_ec_mask = data_chuuchuu["routeType"].isin(["IC", "EC"]) & (data_chuuchuu["country"] == "Switzerland")
data_chuuchuu.loc[sbb_ic_ec_mask, "operator"] = "SBB"
print(f"{sbb_ic_ec_mask.sum()} IC/EC rows in Switzerland assigned 'SBB'")

# IR
sbb_ir_route_numbers = {
    "1651", "1652", "1653", "1654", "1656", "1658", "1662", "1671", "1673", "1674", "1675", "1676", "1677", "1678",
    "1900", "1902", "1904", "1910", "1929", "1943", "1945",
    "2306", "2344", "2354", "2357", "2358", "2361", "2366", "2370", "2374", "2379", "2381", "2388", "2390", "2392", "2393", "2394",
    "3016", "3017", "3022", "3024", "3026", "3029", "3030", "3110",
    "746", "749",
}
sbb_ir_ranges = [(1702, 1843), (1956, 1995), (2055, 2194), (2252, 2292), (2456, 2493), (2503, 2543), (2562, 2599), (2610, 2662), (3251, 3293)]

sbb_ir_mask = (
    (data_chuuchuu["agency"] == "SBB") & (data_chuuchuu["routeType"] == "IR")
    & (data_chuuchuu["routeNumber_trim"].isin(sbb_ir_route_numbers) | in_ranges(data_chuuchuu["routeNumber_int"], sbb_ir_ranges))
)
data_chuuchuu.loc[sbb_ir_mask, "operator"] = "SBB"
print(f"{sbb_ir_mask.sum()} SBB IR rows assigned 'SBB'")

46911 IC/EC rows in Switzerland assigned 'SBB'
44049 SBB IR rows assigned 'SBB'


In [29]:
# R
sbb_r_ranges = [
    (14402, 14693), (17400, 17497), (18940, 18999), (23000, 23535), (24004, 24999),
    (25850, 25864), (26101, 26370), (5607, 5994), (6006, 6758), (7014, 7461), (9901, 9940),
]
sbb_r_mask = (data_chuuchuu["agency"] == "SBB") & (data_chuuchuu["routeType"] == "R") & in_ranges(data_chuuchuu["routeNumber_int"], sbb_r_ranges)
data_chuuchuu.loc[sbb_r_mask, "operator"] = "SBB"
print(f"{sbb_r_mask.sum()} SBB R rows assigned 'SBB'")

# RE
sbb_re_route_numbers = {"2087", "2089", "2090", "2092", "2560", "741", "742", "744"}
sbb_re_ranges = [(18122, 18495), (25500, 25843), (2600, 2607), (3564, 3685), (3958, 3995), (4706, 4716), (4762, 4842), (4908, 4940)]
sbb_re_mask = (
    (data_chuuchuu["agency"] == "SBB") & (data_chuuchuu["routeType"] == "RE")
    & (data_chuuchuu["routeNumber_trim"].isin(sbb_re_route_numbers) | in_ranges(data_chuuchuu["routeNumber_int"], sbb_re_ranges))
)
data_chuuchuu.loc[sbb_re_mask, "operator"] = "SBB"
print(f"{sbb_re_mask.sum()} SBB RE rows assigned 'SBB'")

166177 SBB R rows assigned 'SBB'
26835 SBB RE rows assigned 'SBB'


In [30]:
# S
sbb_s_route_numbers = {
    "30392", "30661", "30698", "30794", "30821", "30823", "30825", "30827", "30829", "30831", "30833", "30835", "30837", "30839",
    "30841", "30843", "30845", "30847", "30849", "30851", "30853", "30855", "30857", "30859", "30861", "30863", "30865", "30867",
    "30869", "30871", "30873", "30949", "30967", "30985",
}
sbb_s_ranges = [
    (14303, 14398),
    (17001, 17099), (17113, 17394), (17525, 17568), (17906, 17943),
    (18011, 18020), (18087, 18093), (18220, 18993),
    (19010, 19293), (19416, 19693), (19921, 19980),
    (20024, 20580),
    (21016, 21395), (21912, 21997),
    (22015, 22175), (22960, 22991),
    (24500, 24697),
    (25100, 25792), (25905, 25993),
    (26018, 26075),
    (7606, 7743), (7819, 7892),
    (8416, 8999),
]
sbb_s_mask = (
    (data_chuuchuu["agency"] == "SBB") & (data_chuuchuu["routeType"] == "S")
    & (data_chuuchuu["routeNumber_trim"].isin(sbb_s_route_numbers) | in_ranges(data_chuuchuu["routeNumber_int"], sbb_s_ranges))
)
data_chuuchuu.loc[sbb_s_mask, "operator"] = "SBB"
print(f"{sbb_s_mask.sum()} SBB S rows assigned 'SBB'")

# SN
sbb_sn_route_numbers = {
    "13702", "13704", "13707", "13709", "13710", "13711", "13712", "13713", "13714", "13715",
    "13716", "13717", "13746", "13748", "13750", "13751", "13752", "13753", "13754", "13755",
    "13756", "13757", "13760", "13762", "13763", "13764", "13765", "13766", "13767", "13769",
    "13770", "13771", "13772", "13773", "13774", "13775", "13776", "13777", "13780", "13781",
    "13782", "13783", "13784", "13785", "13786", "13787", "13790", "13791", "13792", "13793",
    "13794", "13795", "13796", "13797", "13811", "13812", "13813", "13814", "13815", "13816",
    "13818", "13830", "13831", "13832", "13834", "13835", "13836", "13837", "13838", "13839",
    "13840", "13841", "13842", "13843", "13844", "13845", "13846", "13847", "13849",
    "87795", "87797",
}
sbb_sn_mask = (data_chuuchuu["agency"] == "SBB") & (data_chuuchuu["routeType"] == "SN") & data_chuuchuu["routeNumber_trim"].isin(sbb_sn_route_numbers)
data_chuuchuu.loc[sbb_sn_mask, "operator"] = "SBB"
print(f"{sbb_sn_mask.sum()} SBB SN rows assigned 'SBB'")

print("SBB PE / ICE / RB: no operator rule provided -- left as-is")

421310 SBB S rows assigned 'SBB'
2359 SBB SN rows assigned 'SBB'
SBB PE / ICE / RB: no operator rule provided -- left as-is


### OEBB agency

`CJX`/`D`/`ER`/`IR` are always OEBB in Austria. `EC`/`IC` split by `routeNumber` with an OEBB catch-all. `Os` is treated as exclusively Ceske Drahy. `R`/`REX`/`S` are OEBB in Austria except for a few excluded route numbers/ranges (the ones run by other, smaller operators per the observed counts). `RB`/`UEX` have no rule. `WB` is always Westbahn.

In [31]:
oebb_always_mask = data_chuuchuu["routeType"].isin(["CJX", "D", "ER", "IR"]) & (data_chuuchuu["country"] == "Austria")
data_chuuchuu.loc[oebb_always_mask, "operator"] = "OEBB"
print(f"{oebb_always_mask.sum()} CJX/D/ER/IR rows in Austria assigned 'OEBB'")

# EC
oebb_ec_base = (data_chuuchuu["country"] == "Austria") & (data_chuuchuu["routeType"] == "EC")

oebb_ec_oebb_numbers = {
    "100", "102", "106", "114", "140", "141", "142", "143", "144", "145", "146", "147", "148", "149", "164",
    "202", "204", "206", "212", "214", "290", "337", "340", "341", "342", "343", "462", "463", "70", "71", "78", "79",
}
oebb_ec_db_numbers = {"80", "81", "94", "96", "98", "115", "190", "192", "194", "196", "198", "213", "1281"}
oebb_ec_sbb_numbers = {"95", "97", "99", "163", "191", "193", "195", "197", "199"}
oebb_ec_pkp_numbers = {"101", "103", "107", "203", "205", "207"}

data_chuuchuu.loc[oebb_ec_base & data_chuuchuu["routeNumber_trim"].isin(oebb_ec_oebb_numbers), "operator"] = "OEBB"
data_chuuchuu.loc[oebb_ec_base & data_chuuchuu["routeNumber_trim"].isin(oebb_ec_db_numbers), "operator"] = "DB"
data_chuuchuu.loc[oebb_ec_base & data_chuuchuu["routeNumber_trim"].isin(oebb_ec_sbb_numbers), "operator"] = "SBB"
data_chuuchuu.loc[oebb_ec_base & data_chuuchuu["routeNumber_trim"].isin(oebb_ec_pkp_numbers), "operator"] = "PKP Intercity"

oebb_ec_listed_numbers = oebb_ec_oebb_numbers | oebb_ec_db_numbers | oebb_ec_sbb_numbers | oebb_ec_pkp_numbers
oebb_ec_catchall = oebb_ec_base & ~data_chuuchuu["routeNumber_trim"].isin(oebb_ec_listed_numbers)
data_chuuchuu.loc[oebb_ec_catchall, "operator"] = "OEBB"
print(f"{oebb_ec_base.sum()} OEBB EC rows in Austria processed ({oebb_ec_catchall.sum()} via the OEBB catch-all)")

49278 CJX/D/ER/IR rows in Austria assigned 'OEBB'
6398 OEBB EC rows in Austria processed (911 via the OEBB catch-all)


In [32]:
# IC
oebb_ic_base = (data_chuuchuu["country"] == "Austria") & (data_chuuchuu["routeType"] == "IC")

oebb_ic_db_numbers = {"406", "416"}
oebb_ic_pkp_numbers = {"207", "417"}

data_chuuchuu.loc[oebb_ic_base & data_chuuchuu["routeNumber_trim"].isin(oebb_ic_db_numbers), "operator"] = "DB"
data_chuuchuu.loc[oebb_ic_base & data_chuuchuu["routeNumber_trim"].isin(oebb_ic_pkp_numbers), "operator"] = "PKP Intercity"

oebb_ic_listed_numbers = oebb_ic_db_numbers | oebb_ic_pkp_numbers
oebb_ic_catchall = oebb_ic_base & ~data_chuuchuu["routeNumber_trim"].isin(oebb_ic_listed_numbers)
data_chuuchuu.loc[oebb_ic_catchall, "operator"] = "OEBB"
print(f"{oebb_ic_base.sum()} OEBB IC rows in Austria processed ({oebb_ic_catchall.sum()} via the OEBB catch-all)")

# Os -- treated as exclusively Ceske Drahy per the instructions (scoped to the OEBB agency, matching where this was observed)
os_mask = (data_chuuchuu["agency"] == "OEBB") & (data_chuuchuu["routeType"] == "Os")
data_chuuchuu.loc[os_mask, "operator"] = "Ceske Drahy"
print(f"{os_mask.sum()} OEBB 'Os' rows assigned 'Ceske Drahy'")

21737 OEBB IC rows in Austria processed (21262 via the OEBB catch-all)
1905 OEBB 'Os' rows assigned 'Ceske Drahy'


In [33]:
# R
oebb_r_excluded_numbers = {"7807", "1826", "1828"}
oebb_r_mask = (
    (data_chuuchuu["country"] == "Austria") & (data_chuuchuu["routeType"] == "R")
    & ~data_chuuchuu["routeNumber_trim"].isin(oebb_r_excluded_numbers)
    & ~data_chuuchuu["routeNumber_int"].between(8000, 8200)
)
data_chuuchuu.loc[oebb_r_mask, "operator"] = "OEBB"
print(f"{oebb_r_mask.sum()} OEBB R rows in Austria assigned 'OEBB'")
print("OEBB RB: no operator rule provided -- left as-is")

# REX
oebb_rex_excluded = (
    (data_chuuchuu["routeNumber_trim"] == "5572")
    | data_chuuchuu["routeNumber_int"].between(8450, 8600)
    | data_chuuchuu["routeNumber_int"].between(7600, 7900)
)
oebb_rex_mask = (data_chuuchuu["country"] == "Austria") & (data_chuuchuu["routeType"] == "REX") & ~oebb_rex_excluded
data_chuuchuu.loc[oebb_rex_mask, "operator"] = "OEBB"
print(f"{oebb_rex_mask.sum()} OEBB REX rows in Austria assigned 'OEBB'")

25962 OEBB R rows in Austria assigned 'OEBB'
OEBB RB: no operator rule provided -- left as-is
66424 OEBB REX rows in Austria assigned 'OEBB'


In [34]:
# S
oebb_s_excluded_numbers = {
    "5556", "5562", "5564", "5568", "5572", "5574", "5578", "5580", "5584", "5586",
    "5590", "5592", "5596", "5602", "5606", "5694", "25844", "25846",
    "25883", "25885", "25887", "25889", "25891", "25893", "25895", "25897",
}
oebb_s_excluded = (
    data_chuuchuu["routeNumber_int"].between(4350, 4378)
    | data_chuuchuu["routeNumber_int"].between(7350, 7388)
    | data_chuuchuu["routeNumber_int"].between(8000, 8544)
    | data_chuuchuu["routeNumber_trim"].isin(oebb_s_excluded_numbers)
)
oebb_s_mask = (data_chuuchuu["country"] == "Austria") & (data_chuuchuu["routeType"] == "S") & ~oebb_s_excluded
data_chuuchuu.loc[oebb_s_mask, "operator"] = "OEBB"
print(f"{oebb_s_mask.sum()} OEBB S rows in Austria assigned 'OEBB'")
print("OEBB UEX: no operator rule provided -- left as-is")

# WB
wb_mask = data_chuuchuu["routeType"] == "WB"
data_chuuchuu.loc[wb_mask, "operator"] = "Westbahn"
print(f"{wb_mask.sum()} WB rows assigned 'Westbahn'")

212118 OEBB S rows in Austria assigned 'OEBB'
OEBB UEX: no operator rule provided -- left as-is
9901 WB rows assigned 'Westbahn'


### DB agency

`EC`/`ECE` split by `routeNumber`, each with its own catch-all label (`EC`'s catch-all is `'DB'`, `ECE`'s is the more specific `'DB Fernverkehr AG'` -- kept exactly as given rather than harmonized). `GV` is always Govolta. `IC` has no rule yet.

In [35]:
# EC
db_ec_base = (data_chuuchuu["agency"] == "DB") & (data_chuuchuu["routeType"] == "EC")

db_ec_pkp_numbers = {
    "231", "247", "249", "41", "43", "431", "45", "47", "49", "55", "57", "59",
    "230", "246", "248", "40", "42", "430", "44", "46", "48", "54", "56", "58",
}
db_ec_sbb_numbers = {"150", "191", "193", "195", "197", "199", "95", "97", "99", "458", "459"}
db_ec_oebb_numbers = {"213", "115", "114", "212", "290"}

data_chuuchuu.loc[db_ec_base & data_chuuchuu["routeNumber_trim"].isin(db_ec_pkp_numbers), "operator"] = "PKP Intercity"
data_chuuchuu.loc[db_ec_base & data_chuuchuu["routeNumber_trim"].isin(db_ec_sbb_numbers), "operator"] = "SBB"
data_chuuchuu.loc[db_ec_base & data_chuuchuu["routeNumber_trim"].isin(db_ec_oebb_numbers), "operator"] = "OEBB"

db_ec_listed_numbers = db_ec_pkp_numbers | db_ec_sbb_numbers | db_ec_oebb_numbers
db_ec_catchall = db_ec_base & ~data_chuuchuu["routeNumber_trim"].isin(db_ec_listed_numbers)
data_chuuchuu.loc[db_ec_catchall, "operator"] = "DB"
print(f"{db_ec_base.sum()} DB EC rows processed ({db_ec_catchall.sum()} via the 'DB' catch-all)")

# ECE
db_ece_base = (data_chuuchuu["agency"] == "DB") & (data_chuuchuu["routeType"] == "ECE")
db_ece_sbb_numbers = {"190", "192", "194", "196", "198", "94", "96", "98", "151"}

data_chuuchuu.loc[db_ece_base & data_chuuchuu["routeNumber_trim"].isin(db_ece_sbb_numbers), "operator"] = "SBB"
db_ece_catchall = db_ece_base & ~data_chuuchuu["routeNumber_trim"].isin(db_ece_sbb_numbers)
data_chuuchuu.loc[db_ece_catchall, "operator"] = "DB Fernverkehr AG"
print(f"{db_ece_base.sum()} DB ECE rows processed ({db_ece_catchall.sum()} via the 'DB Fernverkehr AG' catch-all)")

# GV
gv_mask = data_chuuchuu["routeType"] == "GV"
data_chuuchuu.loc[gv_mask, "operator"] = "Govolta"
print(f"{gv_mask.sum()} GV rows assigned 'Govolta'")

print("DB IC: no operator rule provided yet -- left as-is")

15914 DB EC rows processed (8818 via the 'DB' catch-all)
2100 DB ECE rows processed (1257 via the 'DB Fernverkehr AG' catch-all)
45 GV rows assigned 'Govolta'
DB IC: no operator rule provided yet -- left as-is


In [36]:
data_chuuchuu["normalized_operator"] = data_chuuchuu["operator"]

db_regio_mask = data_chuuchuu["operator"].str.contains("DB Regio", case=False, na=False)
data_chuuchuu.loc[db_regio_mask, "normalized_operator"] = "DB Regio"
print(f"{db_regio_mask.sum()} rows with operator containing 'DB Regio' assigned 'DB Regio'")

1670040 rows with operator containing 'DB Regio' assigned 'DB Regio'


### Cleanup & summary

In [37]:
try:
    data_chuuchuu = data_chuuchuu.drop(columns=["routeType_norm", "agency_norm", "routeNumber_trim", "routeNumber_int"])
except KeyError:
    pass

print(f"{data_chuuchuu['normalized_operator'].isna().sum()} rows ({data_chuuchuu['normalized_operator'].isna().mean() * 100:.2f}%) still have no normalized_operator")
print()
data_chuuchuu["normalized_operator"].value_counts(dropna=False).head(30)

4845332 rows (32.30%) still have no normalized_operator



normalized_operator
None                               4845332
DB Regio                           1670040
Trenitalia                         1480775
SNCF                               1024148
MAV                                 733201
SBB                                 724215
S Bahn Berlin GmbH                  511104
OEBB                                430488
NS                                  315681
NMBS/SNCB                           254467
dsb-s-tog                           217019
Albtal-Verkehrs-Gesellschaft        216681
Deutsche Bahn                       146489
BLS AG (bls)                        127030
S-Bahn Hamburg                      121756
THURBO                              109620
dsb                                  95029
Hessische Landesbahn                 76582
SNCF VOYAGEURS                       68566
lokaltog                             63537
PKP Intercity                        62851
Arriva                               59672
Südwestdeutsche Verkehrs-AG       

What operators are available for data 2025 ?

In [38]:
agencies_after_2025 = ["PL", "HU", "GTFSDE", "IT", "DK", "SBB", "OEBB", "RENFE"]

before_2025 = ~data_chuuchuu["agency"].isin(agencies_after_2025)
missing_normalized_operator = data_chuuchuu.loc[before_2025, "normalized_operator"].isna()
print(f"{missing_normalized_operator.sum()} rows ({missing_normalized_operator.mean() * 100:.2f}%) still have no normalized_operator (excluding {agencies_after_2025})")
print()
data_chuuchuu.loc[before_2025, "normalized_operator"].value_counts(dropna=False).head(30)

809241 rows (28.16%) still have no normalized_operator (excluding ['PL', 'HU', 'GTFSDE', 'IT', 'DK', 'SBB', 'OEBB', 'RENFE'])



normalized_operator
SNCF                 1022594
None                  809241
NS                    315681
NMBS/SNCB             254467
Deutsche Bahn         103865
OEBB                   72406
SNCF VOYAGEURS         68566
Arriva                 59672
RRReis Arriva          18191
DB                     17726
SBB                    16085
DB Fernverkehr AG      12860
Blauwnet Arriva        12674
Eurostar               11970
R-net Qbuzz             9175
Blauwnet Keolis         7723
RRReis Keolis           6338
R-net NS                5488
Eurobahn                5277
PKP Intercity           4500
Ceske Drahy             4485
NS Int                  4484
NMBS                    4101
SNCF Voyageurs LO       3177
Westbahn                2747
VIAS                    2516
Flixtrain               2469
MAV                     1648
HZ                      1562
Trenitalia              1489
Name: count, dtype: int64

Check which operators are really present (mostly domestic market) for the 2025 data:

In [39]:
op_to_check = "DSB"  # set to "None" (as a string) to check rows with no normalized_operator, or a real value like "OEBB"

if op_to_check == "None":
    operator_mask = data_chuuchuu["normalized_operator"].isna()
else:
    operator_mask = data_chuuchuu["normalized_operator"] == op_to_check

print(data_chuuchuu.loc[before_2025 & operator_mask, "country"].value_counts(dropna=False))
print()
print(data_chuuchuu.loc[before_2025 & operator_mask, "routeType"].value_counts(dropna=False))
print()
print(data_chuuchuu.loc[before_2025 & operator_mask, "journey_type"].value_counts(dropna=False))
print()
print(data_chuuchuu.loc[before_2025 & operator_mask, "agency"].value_counts(dropna=False))

Series([], Name: count, dtype: int64)

Series([], Name: count, dtype: int64)

Series([], Name: count, dtype: int64)

Series([], Name: count, dtype: int64)


In [40]:
op_to_check = "None"  # set to "None" (as a string) to check rows with no normalized_operator, or a real value like "OEBB"

route_type_to_check = "EN"  # this is a routeType value (e.g. "Sprinter", "ICE"), not a journey_type value

if op_to_check == "None":
    operator_mask = data_chuuchuu["normalized_operator"].isna()
else:
    operator_mask = data_chuuchuu["normalized_operator"] == op_to_check

route_type_mask = data_chuuchuu["routeType"] == route_type_to_check

print(data_chuuchuu.loc[before_2025 & operator_mask & route_type_mask, "country"].value_counts(dropna=False))
print()
print(data_chuuchuu.loc[before_2025 & operator_mask & route_type_mask, "journey_type"].value_counts(dropna=False))
print()
print(data_chuuchuu.loc[before_2025 & operator_mask & route_type_mask, "agency"].value_counts(dropna=False))
print()
print(data_chuuchuu.loc[before_2025 & operator_mask & route_type_mask, "routeNumber"].value_counts(dropna=False))

country
Germany    2
Denmark    1
Name: count, dtype: int64

journey_type
international    3
Name: count, dtype: int64

agency
DB    3
Name: count, dtype: int64

routeNumber
93701    3
Name: count, dtype: int64


In [43]:
data_chuuchuu[data_chuuchuu["routeType"] == "OTC"].groupby(["stopName"]).size()

stopName
Aachen HBF (DE)                               5
Angleur                                       5
Aulnoye Aymeries (FR)                        66
Braine-le-Comte / 's-Gravenbrakel             5
Brux.-Midi/Brus.-Zuid                        66
Chaudfontaine                                 5
Chênée                                        5
Creil - Bât Voyageurs                        41
Dolhain-Gileppe                               5
Ecaussinnes                                   5
Familleureux                                  5
Flémalle-Haute                                1
Fraipont                                      5
Hergenrath                                    5
Herstal                                       1
La Louvière-Centre                            5
La Louvière-Sud / La Louvière-Zuid            5
Liers                                         1
Liège-Carré / Luik-Carré                      6
Liège-Guillemins / Luik-Guillemins            6
Liège-Saint-Lambert / Luik-Sint

In [44]:
scope = (data_chuuchuu["agency"] == "IT") & (data_chuuchuu["date"].astype(str).str.startswith("2026-02"))
it_feb = data_chuuchuu.loc[scope]

print(f"{len(it_feb)} rows for agency IT in February 2026")
print(f"{it_feb['journey_id'].nunique()} unique journeys")
print()
print("routeType breakdown:")
print(it_feb["routeType"].value_counts(dropna=False))
print()
print("country breakdown:")
print(it_feb["country"].value_counts(dropna=False))
print()
print("journey_type breakdown:")
print(it_feb["journey_type"].value_counts(dropna=False))
print()
print("normalized_operator breakdown:")
print(it_feb["normalized_operator"].value_counts(dropna=False))

315401 rows for agency IT in February 2026
26318 unique journeys

routeType breakdown:
routeType
REG      289464
MET       10006
FR         6201
IC         5636
Italo      1591
ICN        1001
EC          689
FA          365
FB          303
NCL          46
None         39
IR           32
NJ           17
EXP          11
Name: count, dtype: int64

country breakdown:
country
Italy          314474
Switzerland       801
France            126
Name: count, dtype: int64

journey_type breakdown:
journey_type
domestic         312649
international      2750
unknown               2
Name: count, dtype: int64

normalized_operator breakdown:
normalized_operator
Trenitalia    313065
Italo           1591
None             650
SBB               78
OEBB              17
Name: count, dtype: int64


### Deduplicating trains reported by multiple agencies

Per the data provider: the same physical train can be collected by more than one agency (e.g. a Nightjet through Venice reported by both OEBB as `NJ` and IT as `EN`). A row is uniquely identified by (`routeNumber`, `deutscheBahnStopId`, `plannedArrival`, `plannedDeparture`) -- only one train can have a given number and planned arrival/departure at a given station. Within each duplicate group we keep a single row, preferring (in order):

1. the row where `arrivalCancelled` is true (kept as the most informative one),
2. otherwise the row with the highest `arrivalDelay`,
3. a stable tiebreaker on the original row order (mirrors the SQL's `rowid ASC`).

This mirrors the provider's `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...) = 1` query: a global sort by the same ORDER BY keys, followed by `duplicated(keep="first")` on the partition columns, picks exactly the same row per group.

**Caveat handled below:** pandas' `duplicated()` (like SQL's `PARTITION BY`) treats a shared `NaN` as a match, so two unrelated rows that both happen to be missing `plannedArrival`/`plannedDeparture` would otherwise collapse into the same "group" purely because they share a missing value, not because they're the same train. Rows missing either planned time are therefore excluded from dedup consideration entirely and kept as-is, rather than risking an incorrect merge.

In [23]:
dedup_keys = ["routeNumber", "deutscheBahnStopId", "plannedArrival", "plannedDeparture"]

# only rows where both planned times are known form a reliable dedup key -- pandas' duplicated()
# (like SQL's PARTITION BY) treats a shared NaN as a match, but a missing planned time isn't
# evidence of being the same physical train, so those rows are excluded from dedup consideration
# entirely and kept as-is
has_full_planned_times = data_chuuchuu["plannedArrival"].notna() & data_chuuchuu["plannedDeparture"].notna()
print(f"{(~has_full_planned_times).sum()} rows are missing a planned time and skip deduplication entirely")

# priority 0 = cancelled (kept first, the "most informative" row), 1 = everything else -- mirrors the SQL CASE.
# spelling out the case variants directly (rather than .astype(str).str.lower()) avoids materializing
# a full string-object copy of all 15M rows just to fold case, which was blowing up on memory.
cancelled_priority = (~data_chuuchuu["arrivalCancelled"].isin(["t", "T", "1", "true", "True", "TRUE"])).astype(int)

# an explicit row-order column so the final tiebreaker matches the SQL's `rowid ASC` on the original row order
row_order = np.arange(len(data_chuuchuu))

dedup_sort_key = pd.DataFrame({
    "cancelled_priority": cancelled_priority.to_numpy(),
    "arrivalDelay": data_chuuchuu["arrivalDelay"].to_numpy(),
    "row_order": row_order,
}, index=data_chuuchuu.index)

# a global sort by the same ORDER BY keys as the SQL window function, then duplicated(keep="first")
# on the partition columns picks the same single row per (routeNumber, deutscheBahnStopId,
# plannedArrival, plannedDeparture) group that ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...) = 1 would
sorted_index = dedup_sort_key.loc[has_full_planned_times].sort_values(
    by=["cancelled_priority", "arrivalDelay", "row_order"],
    ascending=[True, False, True],
    na_position="last",
).index

is_duplicate = data_chuuchuu.loc[sorted_index, dedup_keys].duplicated(keep="first")
rows_to_drop = is_duplicate[is_duplicate].index

print(f"{len(rows_to_drop)} duplicate rows removed ({len(rows_to_drop) / len(data_chuuchuu) * 100:.2f}%)")

603521 rows are missing a planned time and skip deduplication entirely
190227 duplicate rows removed (1.27%)


In [24]:
data_chuuchuu = data_chuuchuu.drop(index=rows_to_drop)
# avoid reset_index(drop=True) here: on a frame this wide, it triggers an internal deep copy that
# consolidates every object-dtype column into one contiguous block via np.vstack, which needs a single
# multi-GiB allocation and is what blew up on memory. Reassigning .index directly is metadata-only --
# no data copy, no consolidation -- and produces the same clean 0..N-1 index.
data_chuuchuu.index = np.arange(len(data_chuuchuu))
data_chuuchuu.shape

(14810387, 36)

In [25]:
export_data = input("Export intermediate data to parquet? (y/n): ")

if export_data.lower() == "y":
    os.makedirs(intermediate_outputs_dir, exist_ok=True)

    data_chuuchuu.to_parquet(f"{intermediate_outputs_dir}/data_chuuchuu_{data_selection}_operators.parquet")